# FaceRankNet — Master Colab Notebook
**Feature-Decomposed Facial Beauty Prediction**  
Dataset: SCUT-FBP5500 | Framework: PyTorch + DGL + MediaPipe

---
**Drive folder structure:**
```
MyDrive/Colab Notebooks/FaceRankNet/FaceRankNet/
  ├── config.py, preprocessing.py, dataset.py, model.py ...
  ├── run_colab.ipynb
  └── cache/   <- auto-created, persists across sessions
```
Run cells **in order**.

In [ ]:
import sys, os

# Install DGL without touching torch or numpy (--no-deps)
!pip install dgl --no-deps -f https://data.dgl.ai/wheels/cu121/repo.html -q

# MediaPipe & CV
!pip install -q "mediapipe>=0.10" opencv-python-headless

# Scientific stack
!pip install -q pandas scikit-learn tqdm matplotlib scipy

# DGL dependency
!pip install -q torchdata

# Kaggle dataset downloader
!pip install -q kagglehub

print("Done. Continue to Cell 2.")


In [ ]:

import sys, types

if 'dgl.graphbolt' not in sys.modules:
    _gb = types.ModuleType('dgl.graphbolt')
    _gb.load_graphbolt = lambda: None
    sys.modules['dgl.graphbolt'] = _gb
    print('✓ DGL graphbolt pre-patched')

import dgl, torch
print(f'DGL     : {dgl.__version__}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

# Sanity check: confirm DGL can move a graph to GPU
if torch.cuda.is_available():
    _g = dgl.graph(([0], [1]))
    _g = _g.to('cuda')
    print('✓ DGL CUDA graph test passed')
else:
    print('⚠ No GPU detected — re-check Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# ============================================================
# Cell 3 — Mount Drive, copy project files, download dataset
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil
import pandas as pd
import kagglehub
from sklearn.model_selection import train_test_split

DRIVE_ROOT   = '/content/drive/MyDrive/Colab Notebooks/FaceRankNet/FaceRankNet'
PROJECT_ROOT = '/content/FaceRankNet'
CACHE_DIR    = f'{DRIVE_ROOT}/cache'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(CACHE_DIR,    exist_ok=True)

copied = []
for fname in os.listdir(DRIVE_ROOT):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(DRIVE_ROOT, fname),
                    os.path.join(PROJECT_ROOT, fname))
        copied.append(fname)
sys.path.insert(0, PROJECT_ROOT)
print(f'Copied {len(copied)} files: {copied}')

print('
Downloading SCUT-FBP5500 ...')
KAGGLE_PATH = kagglehub.dataset_download(
    'pranavchandane/scut-fbp5500-v2-facial-beauty-scores')
print('Downloaded to:', KAGGLE_PATH)

IMAGE_DIR   = os.path.join(KAGGLE_PATH, 'Images', 'Images')
LABELS_FILE = os.path.join(KAGGLE_PATH, 'labels.txt')

if not os.path.isdir(IMAGE_DIR):
    for root, _, files in os.walk(KAGGLE_PATH):
        if any(f.lower().endswith(('.jpg','.png')) for f in files):
            IMAGE_DIR = root; break
if not os.path.isfile(LABELS_FILE):
    for root, _, files in os.walk(KAGGLE_PATH):
        if 'labels.txt' in files:
            LABELS_FILE = os.path.join(root,'labels.txt'); break

n_img = len([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg','.png'))])
print(f'Image dir: {IMAGE_DIR}  ({n_img} images)')

df = pd.read_csv(LABELS_FILE, sep=' ', header=None, names=['Filename','Rating'])

# H1: parse ethnicity from filename prefix (A=Asian, C=Caucasian)
df['Ethnicity'] = df['Filename'].apply(
    lambda f: 'Asian' if f[0].upper() == 'A' else 'Caucasian')
print('Ethnicity dist:', df['Ethnicity'].value_counts().to_dict())

df['_b'] = df['Rating'].round(0).astype(int).clip(1,5)
train_df, test_df = train_test_split(df.drop(columns=['_b']),
    test_size=0.20, random_state=42, stratify=df['_b'])
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

TRAIN_CSV     = f'{DRIVE_ROOT}/train_labels.csv'
TEST_CSV      = f'{DRIVE_ROOT}/test_labels.csv'
CACHE_TRAIN   = f'{CACHE_DIR}/train_landmarks.pkl'
CACHE_TEST    = f'{CACHE_DIR}/test_landmarks.pkl'
AVG_FACE_PATH = f'{CACHE_DIR}/avg_face.npy'
PSEUDO_PATH   = f'{CACHE_DIR}/pseudo_labels.pkl'
CHECKPOINT    = f'{DRIVE_ROOT}/checkpoint_best.pt'

train_df.to_csv(TRAIN_CSV, index=False)
test_df.to_csv(TEST_CSV,   index=False)

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print(f'All paths ready. CHECKPOINT: {CHECKPOINT}')


In [ ]:
# ============================================================
# Cell 4 — Extract & cache landmarks  (~20-40 min, saved to Drive)
# Skips extraction jika cache .pkl sudah ada di Drive.
# ============================================================
import os, pickle
from preprocessing import preprocess_dataset

if os.path.exists(CACHE_TRAIN):
    print(f'✓ Cache ditemukan: {CACHE_TRAIN} — load saja (skip extraction)')
    with open(CACHE_TRAIN, 'rb') as f:
        train_coords = pickle.load(f)
else:
    print('Processing TRAIN set ...')
    train_coords = preprocess_dataset(
        image_dir=IMAGE_DIR, csv_path=TRAIN_CSV, cache_path=CACHE_TRAIN)

if os.path.exists(CACHE_TEST):
    print(f'✓ Cache ditemukan: {CACHE_TEST} — load saja (skip extraction)')
    with open(CACHE_TEST, 'rb') as f:
        test_coords = pickle.load(f)
else:
    print('\nProcessing TEST set ...')
    test_coords = preprocess_dataset(
        image_dir=IMAGE_DIR, csv_path=TEST_CSV, cache_path=CACHE_TEST)

print(f'✓ Train: {len(train_coords)} | Test: {len(test_coords)} faces')


In [ ]:
# ============================================================
# Cell 5 — Beauty-Axis Pseudo Labels (H1 — per ethnicity)
# Skips heavy compute jika avg_face.npy + pseudo_labels.pkl sudah ada.
# ============================================================
import os, pickle, numpy as np, pandas as pd
from pseudo_labels import (
    compute_universal_average_face,
    compute_beauty_prototype,
    compute_ethnicity_avg_faces,
    compute_all_pseudo_labels_beauty_axis,
    validate_pseudo_label_quality,
    save_avg_face, save_pseudo_labels,
    load_pseudo_labels, load_avg_face,
)

train_df_loaded = pd.read_csv(TRAIN_CSV)
train_filenames = train_df_loaded["Filename"].tolist()
holistic_map    = dict(zip(train_df_loaded["Filename"],
                           train_df_loaded["Rating"].astype(float)))
ethnicity_map   = dict(zip(train_df_loaded["Filename"],
                           train_df_loaded["Ethnicity"]))

cache_hit = os.path.exists(AVG_FACE_PATH) and os.path.exists(PSEUDO_PATH)

if cache_hit:
    print(f'✓ Cache ditemukan: {AVG_FACE_PATH}')
    print(f'✓ Cache ditemukan: {PSEUDO_PATH}')
    population_mean = load_avg_face(AVG_FACE_PATH)
    pseudo_labels   = load_pseudo_labels(PSEUDO_PATH)
else:
    if 'train_coords' not in globals():
        with open(CACHE_TRAIN, 'rb') as f:
            train_coords = pickle.load(f)

    train_coords_list = [train_coords[f] for f in train_filenames if f in train_coords]
    train_ratings     = [holistic_map[f] for f in train_filenames if f in train_coords]

    # 1) Global population mean & beauty prototype
    population_mean  = compute_universal_average_face(train_coords_list)
    beauty_prototype = compute_beauty_prototype(train_coords_list, train_ratings,
                                                top_k_pct=0.30)

    # 2) Per-ethnicity population means (tanpa top-k filter)
    population_mean_map = compute_ethnicity_avg_faces(
        train_coords, train_filenames, ethnicity_map,
        holistic_ratings=None,
    )

    # 3) Per-ethnicity beauty prototypes (top-30% per ethnicity)
    beauty_prototype_map = compute_ethnicity_avg_faces(
        train_coords, train_filenames, ethnicity_map,
        holistic_ratings=holistic_map,
        top_k_pct=0.30,
    )

    print(f"Per-ethnicity population means: { {k: v.shape for k, v in population_mean_map.items()} }")
    print(f"Per-ethnicity beauty prototypes: { {k: v.shape for k, v in beauty_prototype_map.items()} }")

    # 4) Pseudo-labels via Beauty Axis Projection
    pseudo_labels = compute_all_pseudo_labels_beauty_axis(
        coords_cache=train_coords,
        train_filenames=train_filenames,
        population_mean=population_mean,
        beauty_prototype=beauty_prototype,
        population_mean_map=population_mean_map,
        beauty_prototype_map=beauty_prototype_map,
        ethnicity_map=ethnicity_map,
    )

    # 5) Save
    save_avg_face(population_mean, AVG_FACE_PATH)
    save_pseudo_labels(pseudo_labels, PSEUDO_PATH)

# Diagnostic Spearman rho (cheap — always run)
rho = validate_pseudo_label_quality(pseudo_labels, holistic_map)
print(f'Spearman rho after Beauty Axis: {rho:.4f}')

# Sample sanity check
for sample in ["CF137.jpg", "CF581.jpg", "AF1859.jpg", "AF549.jpg"]:
    if sample in pseudo_labels:
        organs = pseudo_labels[sample]
        mean_score = sum(organs.values()) / len(organs)
        print(f'
{sample}  rating={holistic_map.get(sample, "?"):.3f}  mean_pseudo={mean_score:.3f}')
        for organ, score in organs.items():
            print(f"  {organ:<12}: {score:.3f}")


In [ ]:
# ============================================================
# Cell 6 — Dataset & DataLoader (Step 1: Sampler + Jitter)
# Memakai cache di memory jika sudah dimuat di Cell 4/5, else load dari disk.
# ============================================================
import os, pickle, numpy as np
from pseudo_labels import load_pseudo_labels
from dataset import (
    FaceDataset, PairDataset,
    make_face_loader, make_pair_loader, make_weighted_pair_loader,
)
import config

if 'train_coords' not in globals():
    with open(CACHE_TRAIN, 'rb') as f: train_coords = pickle.load(f)
    print(f'✓ Loaded train_coords from {CACHE_TRAIN}')
if 'test_coords' not in globals():
    with open(CACHE_TEST, 'rb') as f: test_coords = pickle.load(f)
    print(f'✓ Loaded test_coords from {CACHE_TEST}')
if 'pseudo_labels' not in globals():
    pseudo_labels = load_pseudo_labels(PSEUDO_PATH)
    print(f'✓ Loaded pseudo_labels from {PSEUDO_PATH}')
if 'avg_face' not in globals():
    avg_face = np.load(AVG_FACE_PATH)   # (468, 3) — needed for 6-dim node features
    print(f'✓ Loaded avg_face from {AVG_FACE_PATH}')

# Step 1: landmark jitter ON for training set only (test set stays clean).
train_face_ds = FaceDataset(
    TRAIN_CSV, train_coords, pseudo_labels, avg_face=avg_face,
    augment_jitter=True,
)
test_face_ds  = FaceDataset(TEST_CSV,  test_coords,  avg_face=avg_face)
pair_ds       = PairDataset(train_face_ds)

# Step 1: WeightedRandomSampler — rebalances anchor rating buckets.
# Tiap epoch wajah jelek (~5%) dan cantik (~11%) muncul ~4× lebih sering
# di pair_loader → gradient extreme face naik signifikan.
pair_loader   = make_weighted_pair_loader(pair_ds)
val_loader    = make_face_loader(test_face_ds, shuffle=False)

print(f'Train faces: {len(train_face_ds)} | Pairs: {len(pair_ds)} | Test: {len(test_face_ds)}')

# Diagnostic: confirm sampler is actually rebalancing
import numpy as np
ratings = np.array(train_face_ds.ratings)
pair_anchor_ratings = np.array([ratings[a] for a, _ in pair_ds._pairs])
bins = [0, 2, 3, 4, 5.1]
counts_raw = np.histogram(pair_anchor_ratings, bins=bins)[0]
print(f'Pair anchor rating distribution (raw): {counts_raw.tolist()}')
print(f'  Sampler weights this to roughly equal frequency per epoch.')

batch_a, batch_b, mask = next(iter(pair_loader))
print('Batch A ratings shape:', batch_a['ratings'].shape)
print('Batch A ratings sample:', batch_a['ratings'].numpy().round(2).tolist())


In [ ]:
# ============================================================
# Cell 7 — Model instantiation + dry-run
# ============================================================
import torch
from model import FaceRankNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = FaceRankNet().to(device)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

# Move subgraphs to device — requires CUDA-enabled DGL
model.eval()
with torch.no_grad():
    sg  = {k: v.to(device) for k, v in batch_a['subgraphs'].items()}
    out = model(sg)
print('global_score shape:', out['global_score'].shape)
print('organ_weights     :', out['organ_weights'].cpu().numpy().round(3))

In [ ]:
PROJECT_ROOT = '/content/FaceRankNet'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ============================================================
# Cell 8 — Training  (resume dari checkpoint jika ada)
# ============================================================
import os
from train import train as run_training

# Konfirmasi status checkpoint sebelum training
if os.path.exists(CHECKPOINT):
    import torch
    ckpt = torch.load(CHECKPOINT, map_location='cpu')
    print(f"✓ Checkpoint ditemukan — epoch {ckpt['epoch']}, best PCC={ckpt['best_pcc']:.4f}")
    print(f"  λ_rank saat itu: {ckpt.get('lambdas', ['?','?'])[1]:.4f}")
    print(f"  → Lanjut dari epoch {ckpt['epoch'] + 1}")
else:
    print("⚠ Checkpoint tidak ditemukan — training dari awal")

trained_model = run_training(
    train_csv=TRAIN_CSV,
    test_csv=TEST_CSV,
    landmark_cache_train=CACHE_TRAIN,
    landmark_cache_test=CACHE_TEST,
    pseudo_labels_path=PSEUDO_PATH,
    avg_face_path=AVG_FACE_PATH,
    checkpoint_path=CHECKPOINT,
    resume=True,
)
print("\nTraining complete.")


In [ ]:
# ============================================================
# Cell 8b — Load checkpoint TANPA training  (SKIP Cell 8)
# Gunakan cell ini jika checkpoint sudah ada di Drive dan
# kamu hanya ingin evaluasi / visualisasi.
# ============================================================
import os, torch
from model import FaceRankNet

if not os.path.exists(CHECKPOINT):
    raise FileNotFoundError(
        f"Checkpoint tidak ditemukan di:\n  {CHECKPOINT}\n"
        "Upload checkpoint_best.pt ke Google Drive terlebih dahulu."
    )

ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=False)
print(f"Checkpoint info:")
print(f"  epoch    : {ckpt['epoch']}")
print(f"  best PCC : {ckpt['best_pcc']:.4f}")
print(f"  lambdas  : {[round(l, 4) for l in ckpt.get('lambdas', [])]}")

trained_model = FaceRankNet().to(device)
trained_model.load_state_dict(ckpt["model_state_dict"])
trained_model.eval()
print(f"\n✓ Model loaded → eval mode  (device={device})")

In [ ]:
# ============================================================
# Cell 9 — Evaluation: PCC, MAE, DPD
# ============================================================
from evaluate import run_full_evaluation, validate_local_scores

trained_model.eval()
metrics = run_full_evaluation(trained_model, val_loader, device)
print('=' * 40)
print(f'  PCC : {metrics["pcc"]:.4f}')
print(f'  MAE : {metrics["mae"]:.4f}')
print(f'  DPD : {metrics["dpd"]:.4f}')
print('=' * 40)
valid = validate_local_scores(trained_model, train_face_ds, device)
print('Local scores:', 'PASS' if valid else 'FAIL')

In [ ]:
# ============================================================
# Cell 10 — Visualisation: organ bar chart + cross-organ attention heatmap
# ============================================================
import torch, matplotlib.pyplot as plt, matplotlib, numpy as np
from dataset import collate_faces
from torch.utils.data import DataLoader

matplotlib.rcParams['figure.dpi'] = 120
colors = ['#4C72B0','#55A868','#C44E52','#8172B2','#CCB974']

sample_batch = next(iter(DataLoader(
    test_face_ds, batch_size=4, shuffle=False, collate_fn=collate_faces)))

trained_model.eval()
with torch.no_grad():
    sg  = {k: v.to(device) for k, v in sample_batch['subgraphs'].items()}
    out = trained_model(sg)

organ_names = list(out['local_scores'].keys())
n = len(sample_batch['filenames'])

# ── Fig 1: Organ score bar chart ────────────────────────────────────────────
fig, axes = plt.subplots(1, n, figsize=(4*n, 4), sharey=True)
if n == 1: axes = [axes]
for i, ax in enumerate(axes):
    scores = [out['local_scores'][o][i].item() for o in organ_names]
    ax.bar(organ_names, scores, color=colors, edgecolor='white')
    ax.axhline(out['global_score'][i].item(), color='black', ls='--', lw=1.2,
               label=f'Global={out["global_score"][i].item():.2f}')
    ax.axhline(sample_batch['ratings'][i].item(), color='red', ls=':', lw=1.2,
               label=f'GT={sample_batch["ratings"][i].item():.2f}')
    ax.set_ylim(1, 5); ax.set_title(sample_batch['filenames'][i], fontsize=7)
    ax.set_xticklabels([o.replace('_','\n') for o in organ_names], fontsize=7)
    ax.legend(fontsize=6)
fig.suptitle('FaceRankNet - Organ Scores', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/organ_scores_bar.png', bbox_inches='tight')
plt.show()

# ── Fig 2: Cross-organ attention heatmap (real learned importance) ──────────
# attn_weights: (B, 5, 5) — averaged across batch & heads
attn = out['attn_weights'].cpu().numpy()   # (B, num_heads*5, 5) or (B, 5, 5)
attn_mean = attn.mean(axis=0)              # (5, 5) — mean across batch

short_names = [o.replace('_eye','_e').replace('jawline','jaw') for o in organ_names]
fig2, ax2 = plt.subplots(figsize=(6, 5))
im = ax2.imshow(attn_mean, cmap='YlOrRd', vmin=0, vmax=attn_mean.max())
ax2.set_xticks(range(len(organ_names))); ax2.set_xticklabels(short_names, fontsize=9)
ax2.set_yticks(range(len(organ_names))); ax2.set_yticklabels(short_names, fontsize=9)
ax2.set_xlabel('Key (organ yang diperhatikan)'); ax2.set_ylabel('Query (organ yang memperhatikan)')
for i in range(len(organ_names)):
    for j in range(len(organ_names)):
        ax2.text(j, i, f'{attn_mean[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax2)
ax2.set_title('Cross-Organ Attention (rata-rata batch)\nInilah organ importance yang real', fontsize=10)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/cross_organ_attention.png', bbox_inches='tight', dpi=150)
plt.show()

# ── Fig 3: Row-sum = effective attention received per organ ──────────────────
received = attn_mean.sum(axis=0)   # berapa banyak "attention" diterima tiap organ
fig3, ax3 = plt.subplots(figsize=(6, 3))
ax3.bar(short_names, received / received.sum(), color=colors, edgecolor='white')
ax3.set_ylabel('Attention share'); ax3.set_title('Effective Organ Importance (dari attention)', fontsize=10)
ax3.set_ylim(0, 0.5)
for i, v in enumerate(received / received.sum()):
    ax3.text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/organ_importance_attention.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figures saved to Drive.')

In [ ]:
# ============================================================
# Cell 11 — Extended Evaluation
# 1. Scatter predicted vs GT
# 2. Error distribution per rating bucket
# 3. PCC/MAE per ethnicity (H1 check)
# 4. Score distribution KDE
# 5. Top-K ranking precision
# 6. Full test-set attention heatmap
# ============================================================
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib
from scipy import stats
from torch.utils.data import DataLoader
from dataset import collate_faces

matplotlib.rcParams['figure.dpi'] = 110

# ── Collect all predictions on test set ─────────────────────────────────────
trained_model.eval()
all_preds, all_gts, all_files = [], [], []
all_attn = []

with torch.no_grad():
    for batch in DataLoader(test_face_ds, batch_size=64, shuffle=False, collate_fn=collate_faces):
        sg  = {k: v.to(device) for k, v in batch['subgraphs'].items()}
        out = trained_model(sg)
        all_preds.extend(out['global_score'].cpu().numpy())
        all_gts.extend(batch['ratings'].numpy())
        all_files.extend(batch['filenames'])
        all_attn.append(out['attn_weights'].cpu().numpy())   # (B, 5, 5)

preds = np.array(all_preds)
gts   = np.array(all_gts)
files = np.array(all_files)
errors = preds - gts

# ── 1. Scatter plot: Predicted vs GT ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(gts, preds, alpha=0.35, s=15, color='steelblue', edgecolors='none')
lims = [1, 5]
ax.plot(lims, lims, 'r--', lw=1.2, label='Perfect prediction')
# Regression line
m, b, r, p, _ = stats.linregress(gts, preds)
xs = np.linspace(1, 5, 100)
ax.plot(xs, m*xs + b, 'orange', lw=1.5, label=f'Fit (r={r:.3f})')
ax.set_xlabel('GT Rating'); ax.set_ylabel('Predicted Score')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_title(f'Predicted vs GT  (PCC={np.corrcoef(gts, preds)[0,1]:.4f})', fontsize=11)
ax.legend(); plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/scatter_pred_vs_gt.png', bbox_inches='tight', dpi=150)
plt.show()

# ── 2. Error distribution per rating bucket ──────────────────────────────────
buckets = {'Jelek (1–2)': (1, 2), 'Rata-rata (2–3)': (2, 3),
           'Cukup (3–4)': (3, 4), 'Cantik (4–5)': (4, 5)}
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
colors4 = ['#C44E52','#8172B2','#4C72B0','#55A868']
for ax, (label, (lo, hi)), c in zip(axes, buckets.items(), colors4):
    mask = (gts >= lo) & (gts < hi)
    err  = errors[mask]
    ax.hist(err, bins=20, color=c, alpha=0.8, edgecolor='white')
    ax.axvline(0, color='black', lw=1.2, ls='--')
    ax.axvline(err.mean(), color='red', lw=1.2, ls='-', label=f'mean={err.mean():.2f}')
    ax.set_title(f'{label}\n(n={mask.sum()})', fontsize=9)
    ax.set_xlabel('Error (pred − GT)'); ax.legend(fontsize=8)
fig.suptitle('Error Distribution per Rating Bucket', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/error_per_bucket.png', bbox_inches='tight', dpi=150)
plt.show()

# ── 3. PCC & MAE per ethnicity (H1 evaluation) ───────────────────────────────
test_df_loaded = pd.read_csv(TEST_CSV)
eth_map = dict(zip(test_df_loaded['Filename'], test_df_loaded['Ethnicity']))
ethnicities = {'Asian': [], 'Caucasian': []}
for f, p, g in zip(files, preds, gts):
    eth = eth_map.get(f, 'Unknown')
    if eth in ethnicities:
        ethnicities[eth].append((p, g))

eth_metrics = {}
for eth, vals in ethnicities.items():
    if not vals: continue
    ps, gs = zip(*vals)
    ps, gs = np.array(ps), np.array(gs)
    eth_metrics[eth] = {
        'PCC': float(np.corrcoef(ps, gs)[0,1]),
        'MAE': float(np.abs(ps - gs).mean()),
        'n'  : len(ps)
    }

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
eth_names = list(eth_metrics.keys())
pccs = [eth_metrics[e]['PCC'] for e in eth_names]
maes = [eth_metrics[e]['MAE'] for e in eth_names]
ns   = [eth_metrics[e]['n']   for e in eth_names]

axes[0].bar(eth_names, pccs, color=['#4C72B0','#C44E52'])
axes[0].set_ylim(0, 1); axes[0].set_title('PCC per Ethnicity')
axes[0].axhline(0.70, color='gray', ls='--', lw=1, label='Target 0.70')
for i, (v, n) in enumerate(zip(pccs, ns)):
    axes[0].text(i, v+0.01, f'{v:.3f}\n(n={n})', ha='center', fontsize=9)
axes[0].legend()

axes[1].bar(eth_names, maes, color=['#4C72B0','#C44E52'])
axes[1].set_title('MAE per Ethnicity')
axes[1].axhline(0.36, color='gray', ls='--', lw=1, label='Target 0.36')
for i, (v, n) in enumerate(zip(maes, ns)):
    axes[1].text(i, v+0.003, f'{v:.3f}\n(n={n})', ha='center', fontsize=9)
axes[1].legend()

fig.suptitle('Per-Ethnicity Performance (H1 Validation)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/ethnicity_metrics.png', bbox_inches='tight', dpi=150)
plt.show()

# ── 4. Score distribution KDE ────────────────────────────────────────────────
from scipy.stats import gaussian_kde
fig, ax = plt.subplots(figsize=(7, 4))
xs = np.linspace(1, 5, 300)
for arr, label, c in [(gts, 'GT (holistic)', 'steelblue'), (preds, 'Predicted', 'orangered')]:
    kde = gaussian_kde(arr, bw_method=0.3)
    ax.plot(xs, kde(xs), label=label, lw=2, color=c)
    ax.fill_between(xs, kde(xs), alpha=0.15, color=c)
ax.set_xlabel('Score'); ax.set_ylabel('Density')
ax.set_title('Score Distribution: GT vs Predicted', fontsize=11)
ax.legend(); plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/score_distribution_kde.png', bbox_inches='tight', dpi=150)
plt.show()

# ── 5. Top-K ranking precision ───────────────────────────────────────────────
# "Berapa % wajah cantik (GT top-K) yang juga ada di prediksi top-K?"
print("\nTop-K Ranking Precision:")
print(f"{'K':>6} | {'Precision':>10} | {'Recall':>8}")
print("-" * 30)
gt_rank   = np.argsort(-gts)
pred_rank = np.argsort(-preds)
for k in [10, 25, 50, 100]:
    top_gt   = set(gt_rank[:k])
    top_pred = set(pred_rank[:k])
    prec = len(top_gt & top_pred) / k
    rec  = len(top_gt & top_pred) / k
    print(f"{k:>6} | {prec:>10.1%} | {rec:>8.1%}")

# ── 6. Full test-set attention heatmap ──────────────────────────────────────
organ_names = ['left_eye','right_eye','nose','mouth','jawline']
short_names  = ['l_eye','r_eye','nose','mouth','jaw']
attn_all = np.concatenate(all_attn, axis=0)   # (N, 5, 5)
attn_mean = attn_all.mean(axis=0)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn_mean, cmap='YlOrRd')
ax.set_xticks(range(5)); ax.set_xticklabels(short_names, fontsize=9)
ax.set_yticks(range(5)); ax.set_yticklabels(short_names, fontsize=9)
ax.set_xlabel('Key (diperhatikan)'); ax.set_ylabel('Query (yang memperhatikan)')
for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{attn_mean[i,j]:.3f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax)
ax.set_title(f'Cross-Organ Attention  (N={len(attn_all)} test faces)', fontsize=10)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/attention_fulltest.png', bbox_inches='tight', dpi=150)
plt.show()
print('\nSemua figure tersimpan ke Drive.')